# 01 · Models, messages and prompts

After this notebook you can create a LangChain chat model, talk to it with typed messages, build reusable prompt
templates (variables, a history slot, few-shot examples, partials), read token usage off every response, and say
exactly what LangChain adds on top of the raw OpenAI SDK.

**Time:** ~20 min · **Runs:** offline on CPU in < 1 min · **Needs:** [OpenAI API basics](../08_llm_apps/01_openai_api_basics.ipynb)

## Why this matters in interviews

- "Why use LangChain at all when the provider already has an SDK?" is asked in almost every GenAI loop. You need a
  concrete answer — one model interface, typed messages, templates, batching and streaming for free — **and** the cost:
  a layer between you and the exact prompt.
- Prompt templates are the unit you version, test and review. Interviewers want to hear "every prompt goes through a
  template", not f-strings scattered through the code.
- Roles (system / user / assistant / tool) and the tool-call rules are where most "my agent crashed with a 400" bugs live.
- Every response carries token usage. Knowing where to read it — and why a chat gets more expensive every turn — is
  how you answer cost questions with numbers.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `pd10` | Why does LangChain (or any framework) exist, if the providers already have APIs? |
| `tl02` | LangChain, LangGraph and LlamaIndex — what is the difference? |
| `pr15` | What is the difference between the system, user and assistant roles, and does it matter? |
| `pr13` | What is a prompt template and why should everything go through one? |
| `fo13` | What is zero-shot, few-shot and chain-of-thought prompting? |
| `pr03` | How many few-shot examples should you use, and which ones? |
| `in08` | What is streaming, and what does it actually improve? |
| `in18` | How do you calculate cost per request, properly? |
| `pd26` | What is the cost of a multi-turn conversation, and why does it surprise people? |

## Setup

Every notebook in this set talks to a model through `llm_setup.py`. With `OPENAI_API_KEY` set you get real
`gpt-4.1-mini` answers; with Ollama running you get a free local model; with neither, a deterministic **offline
stand-in** answers so every cell still runs (its answers are canned, but the plumbing is real).

The code you would write in a real project is just:

```python
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)   # reads OPENAI_API_KEY from the environment
```

`langchain_chat_model(**kw)` returns exactly that `ChatOpenAI`, plus the `base_url` / `http_client` that point it at
Ollama or the stand-in. A provider-agnostic alternative is `from langchain.chat_models import init_chat_model` then
`init_chat_model("openai:gpt-4.1-mini")` (or `"anthropic:..."`, `"ollama:..."`) — same object, chosen by a string.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER
from llm_setup import langchain_chat_model, OFFLINE

import json, time
import matplotlib.pyplot as plt

llm = langchain_chat_model(temperature=0)
print("LangChain class:", type(llm).__module__ + "." + type(llm).__name__)
print("model          :", llm.model_name, "| temperature:", llm.temperature)

## 1. The LangChain 1.x package map (and what moved)

**In plain English:** "LangChain" is now a family of small packages. The interfaces live in `langchain-core`, each
provider has its own package, and the old all-in-one classes were moved to `langchain-classic`.

| Package | What is in it | You import |
|---|---|---|
| `langchain-core` | the interfaces: messages, prompts, Runnables (LCEL), tools, documents, vector-store base | `from langchain_core.prompts import ChatPromptTemplate` |
| `langchain-openai` | `ChatOpenAI`, `OpenAIEmbeddings` (one package per provider: `langchain-anthropic`, `langchain-ollama`, ...) | `from langchain_openai import ChatOpenAI` |
| `langchain` (1.x) | the agent API: `create_agent`, middleware, `init_chat_model`; re-exports messages/tools | `from langchain.agents import create_agent` |
| `langchain-text-splitters` | chunking | `from langchain_text_splitters import RecursiveCharacterTextSplitter` |
| `langchain-community` | third-party loaders, FAISS, BM25 (now being sunset in favour of standalone packages) | `from langchain_community.vectorstores import FAISS` |
| `langchain-classic` | the 0.x legacy: `LLMChain`, `RetrievalQA`, `AgentExecutor`, `ConversationBufferMemory`, `EnsembleRetriever` | `from langchain_classic.retrievers import EnsembleRetriever` |
| `langgraph` | the runtime under `create_agent`: state graphs, checkpointers, interrupts | `from langgraph.checkpoint.memory import InMemorySaver` |

Most tutorials online are 0.x. The next cell proves which old imports no longer work in this environment.

In [ ]:
import importlib
import importlib.metadata as md

for pkg in ["langchain", "langchain-core", "langchain-openai", "langchain-community", "langchain-classic", "langgraph"]:
    print(f"{pkg:22s} {md.version(pkg)}")

old_imports = [  # (0.x module, name, where it lives now)
    ("langchain.chat_models", "ChatOpenAI", "langchain_openai"),
    ("langchain.prompts", "ChatPromptTemplate", "langchain_core.prompts"),
    ("langchain.chains", "LLMChain", "langchain_classic.chains  (or just: prompt | llm)"),
    ("langchain.agents", "AgentExecutor", "langchain_classic.agents  (new API: create_agent)"),
    ("langchain.memory", "ConversationBufferMemory", "langchain_classic.memory (new: a checkpointer)"),
    ("langchain.retrievers", "EnsembleRetriever", "langchain_classic.retrievers"),
]
print()
for mod, name, new_home in old_imports:
    try:
        getattr(importlib.import_module(mod), name)
        status = "still works"
    except (ImportError, AttributeError):
        status = "FAILS"
    print(f"{'from ' + mod + ' import ' + name:54s} -> {status} | now: {new_home}")
    assert status == "FAILS"

## 2. Your first call: `invoke` returns an `AIMessage`

A chat model is a **Runnable**: it has `invoke` (one input), `batch` (many) and `stream` (tokens as they arrive).
The input can be a plain string (treated as one user message) or a list of messages. The output is an `AIMessage`.

In [ ]:
ai = llm.invoke("What is 17*23?")

print("type        :", type(ai).__name__)
print("content     :", ai.content)
print("text        :", ai.text)            # same thing as a plain string, even when content is a list of blocks
print("content_blocks:", ai.content_blocks)  # provider-neutral typed blocks (text, reasoning, tool_call, image...)
assert "391" in ai.content

## 3. Messages and roles

| Message class | Role on the wire | Who writes it | Use it for |
|---|---|---|---|
| `SystemMessage` | `system` (or `developer`) | you | standing instructions: persona, rules, output format |
| `HumanMessage` | `user` | the user | the request (text, images, files) |
| `AIMessage` | `assistant` | the model | its reply — text and/or `tool_calls` |
| `ToolMessage` | `tool` | your code | the result of running one tool call, linked by `tool_call_id` |

You can pass messages three ways — objects, `(role, text)` tuples, or OpenAI-style dicts. They are converted to the
same thing, so the three calls below must give the same answer. `convert_to_openai_messages` then shows the JSON that
actually goes over the wire — the first thing to print when you debug a chain: **always be able to see the exact prompt.**

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage, convert_to_openai_messages

system = "Classify the sentiment of the review as one of: positive, negative, neutral."
review = "Review: 'The screen is terrible and slow.'"

as_objects = [SystemMessage(system), HumanMessage(review)]
as_tuples = [("system", system), ("human", review)]
as_dicts = [{"role": "system", "content": system}, {"role": "user", "content": review}]

answers = [llm.invoke(m).content for m in (as_objects, as_tuples, as_dicts)]
for form, a in zip(["objects", "tuples", "dicts"], answers):
    print(f"{form:8s} -> {a}")
assert len(set(answers)) == 1, "all three forms are the same request"

print("\non the wire:", json.dumps(convert_to_openai_messages(as_objects), indent=2))

### `ToolMessage` and the tool-call rule

When the model wants a tool, it returns an `AIMessage` whose `tool_calls` list has a name, arguments and an `id`.
Your code runs the tool and answers with a `ToolMessage(content=..., tool_call_id=that_id)`. The API enforces the
pairing: **every tool call id must be answered before the next user or assistant message**. Notebook 03 builds the
full loop; here is the shape, and the 400 you get when you break the rule.

In [ ]:
import openai

tool_call = {"name": "calculator", "args": {"expression": "17*23"}, "id": "call_1"}
conversation = [
    HumanMessage("What is 17*23?"),
    AIMessage("", tool_calls=[tool_call]),        # the model asked for a tool
    ToolMessage("391", tool_call_id="call_1"),    # your code answered it
]
print("final answer:", llm.invoke(conversation).content)

broken = conversation[:2] + [HumanMessage("thanks")]   # the ToolMessage is missing
try:
    llm.invoke(broken)
except openai.BadRequestError as e:
    print("\nBadRequestError", e.status_code, "->", e.body["message"][:120], "...")

### The model is stateless — "memory" is a list you resend

**In plain English:** the model remembers nothing between calls. A chatbot "remembers" because your code sends the
whole conversation again every time.

That has a cost that surprises people (`pd26`): the input grows every turn, so the **total** tokens you pay for grow
roughly with the square of the number of turns. `llm.get_num_tokens_from_messages` counts tokens locally with
`tiktoken` — no API call — so you can estimate before you spend (second cell).

In [ ]:
print("no history  :", llm.invoke("What is my name?").content)

history = [
    HumanMessage("My name is Sujan."),
    AIMessage("Nice to meet you, Sujan!"),
    HumanMessage("What is my name?"),
]
reply = llm.invoke(history).content
print("with history:", reply)
assert "Sujan" in reply

In [ ]:
turn_text = "Can you explain how the refund process works for an order that arrived damaged?"
reply_text = "Sure. Open the order, choose Report a problem, attach a photo and pick Refund. It takes 5 to 7 days."

msgs, per_call = [SystemMessage("You are a helpful support assistant.")], []
for turn in range(1, 21):
    msgs.append(HumanMessage(f"(turn {turn}) {turn_text}"))
    per_call.append(llm.get_num_tokens_from_messages(msgs))   # what this call would send
    msgs.append(AIMessage(reply_text))

cumulative = [sum(per_call[:i + 1]) for i in range(len(per_call))]
print("input tokens, call 1 / call 20:", per_call[0], "/", per_call[-1])
print("total input tokens paid over 20 turns:", cumulative[-1])
assert all(b > a for a, b in zip(per_call, per_call[1:])), "every turn sends more than the last"

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(range(1, 21), per_call, marker="o"); ax[0].set_title("Input tokens per call (grows linearly)")
ax[1].plot(range(1, 21), cumulative, marker="o", color="tab:red"); ax[1].set_title("Total input tokens paid (grows ~quadratically)")
for a in ax:
    a.set_xlabel("turn"); a.set_ylabel("tokens"); a.set_xticks(range(0, 21, 5))
plt.tight_layout(); plt.show()

## 4. `invoke`, `batch`, `stream`

- `batch(list)` runs the calls concurrently (a thread pool) and returns results **in input order**.
- `stream(input)` yields `AIMessageChunk`s. Chunks support `+`, so adding them gives the full message.

Streaming does not make the model faster — the last token arrives at the same time either way. What improves is
**time to first token**: the user sees words almost immediately instead of staring at a spinner. To make that visible
offline, the second cell gives the stand-in a network delay and a per-token delay (real models have both anyway).
`stream_usage=True` asks for token counts in the final chunk (on by default when talking to api.openai.com directly).

In [ ]:
questions = ["What is 2+3?", "What is 10*10?", "What is 7-9?"]
for q, a in zip(questions, llm.batch(questions)):
    print(f"{q:15s} -> {a.content}")

In [ ]:
if OFFLINE:
    OFFLINE.latency, OFFLINE.token_delay = 0.3, 0.03     # simulate a network round-trip + generation speed

text = ("Summarize this for a new engineer.\n\n"
        "LangChain gives every chat model the same interface. You call invoke for one answer, batch for many "
        "and stream for tokens as they arrive. Streaming does not make the model faster, but the first words "
        "show up sooner.")
t0, arrivals, full = time.perf_counter(), [], None
for chunk in llm.stream(text, stream_usage=True):
    full = chunk if full is None else full + chunk
    if chunk.content:
        arrivals.append((time.perf_counter() - t0, len(full.content)))

if OFFLINE:
    OFFLINE.latency, OFFLINE.token_delay = 0.0, 0.0

ttft, total = arrivals[0][0], arrivals[-1][0]
print("full text :", full.content)
print("usage     :", full.usage_metadata)
print(f"time to first token {ttft:.2f}s | last token {total:.2f}s")
assert ttft < total

xs, ys = zip(*arrivals)
plt.figure(figsize=(8, 3.4))
plt.step(xs, ys, where="post", label="stream(): text on screen")
plt.axvline(total, color="tab:red", ls="--", label="invoke(): whole reply appears only here")
plt.axvline(ttft, color="tab:green", ls=":", label=f"first token at {ttft:.2f}s")
plt.title("Streaming improves time to first token, not total time")
plt.xlim(0, total * 1.05)
plt.xlabel("seconds since request"); plt.ylabel("characters received"); plt.legend(loc="upper left")
plt.tight_layout(); plt.show()

## 5. What comes back: `usage_metadata` and `response_metadata`

| Field | What it holds | Use it for |
|---|---|---|
| `ai.usage_metadata` | `input_tokens`, `output_tokens`, `total_tokens`, plus details (cached input, reasoning tokens) — **same keys for every provider** | cost, budgets, dashboards |
| `ai.response_metadata` | provider-specific: `model_name`, `finish_reason`, `system_fingerprint`, `id`, raw `token_usage` | debugging, logging which model version answered |
| `ai.id` | LangChain's run id for this message | joining traces |

Cost per request = input tokens × input price + output tokens × output price (and cached input tokens at their own,
cheaper price). Prices below are the published gpt-4.1-mini list prices at the time of writing — check the pricing page.

In [ ]:
ai = llm.invoke("What is 12*12?")
print("usage_metadata   :", ai.usage_metadata)
print("finish_reason    :", ai.response_metadata["finish_reason"])
print("model_name       :", ai.response_metadata["model_name"])

PRICE_PER_M = {"gpt-4.1-mini": {"input": 0.40, "cached": 0.10, "output": 1.60}}   # USD per 1M tokens

def cost_usd(usage: dict, model: str = "gpt-4.1-mini") -> float:
    p = PRICE_PER_M[model]
    cached = usage.get("input_token_details", {}).get("cache_read", 0) or 0
    fresh = usage["input_tokens"] - cached
    return (fresh * p["input"] + cached * p["cached"] + usage["output_tokens"] * p["output"]) / 1e6

print(f"cost of this call: ${cost_usd(ai.usage_metadata):.8f}")
print(f"20-turn chat above (input only): ${cumulative[-1] * PRICE_PER_M['gpt-4.1-mini']['input'] / 1e6:.6f}")

### Model parameters

| Parameter | What it does | Notes |
|---|---|---|
| `model` | which model | `llm.profile` lists its limits (context window, tool support) |
| `temperature` | randomness of sampling | `0` for extraction/classification. GPT-5-family reasoning models **reject** it — use `reasoning_effort` |
| `max_tokens` | cap on output tokens | hitting it gives `finish_reason == "length"` — a truncated answer, check for it |
| `timeout` | seconds before giving up | set it; the OpenAI SDK default read timeout is 600 s |
| `max_retries` | automatic retries on 429 / 5xx | the OpenAI SDK default is 2 |
| `seed`, `top_p`, `stop`, `logprobs` | reproducibility, nucleus sampling, stop strings, token probabilities | passed straight to the API |

You can set them on the constructor, or per call: `llm.invoke(x, max_tokens=5)`, or `llm.bind(max_tokens=5)`.

In [ ]:
short = langchain_chat_model(temperature=0, max_tokens=6, timeout=30, max_retries=2)
cut = short.invoke("Summarize: The quick brown fox jumps over the lazy dog. It was a sunny day in the park.")
print("reply        :", repr(cut.content))
print("finish_reason:", cut.response_metadata["finish_reason"])
assert cut.response_metadata["finish_reason"] == "length"

print("\ncontext window (llm.profile):", llm.profile["max_input_tokens"], "tokens in,",
      llm.profile["max_output_tokens"], "out | tool calling:", llm.profile["tool_calling"])

## 6. Prompt templates

**In plain English:** a prompt template is a prompt with blanks. You write it once, fill the blanks per request, and
version it like code.

`ChatPromptTemplate.from_messages` takes `(role, template)` pairs. `{name}` marks a variable. Invoking the template
returns a `ChatPromptValue` — the list of messages the model will see. Two more things (second cell): literal braces,
for example a JSON example inside the prompt, must be doubled `{{ }}`; and a template pipes straight into a model —
`prompt | llm` is a chain (notebook 02 is all about that operator).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

support_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a support assistant for {company}. Answer in {language}, in at most {n} sentences."),
    ("human", "{question}"),
])
print("input_variables:", support_prompt.input_variables)

value = support_prompt.invoke({"company": "Acme", "language": "English", "n": 2, "question": "What is 6*7?"})
print(type(value).__name__)
for m in value.to_messages():
    print(f"  {type(m).__name__:14s} {m.content}")

try:
    support_prompt.invoke({"company": "Acme", "question": "hi"})
except KeyError as e:
    print("\nmissing variables ->", str(e)[:105], "...")

In [ ]:
json_prompt = ChatPromptTemplate.from_messages([("human", 'Reply as JSON like {{"answer": 1}}. Question: {q}')])
print("variables:", json_prompt.input_variables)             # only q - the doubled braces are literal
print(json_prompt.invoke({"q": "What is 2+2?"}).to_messages()[0].content)

chain = support_prompt | llm
print("\nchain answer:", chain.invoke({"company": "Acme", "language": "English", "n": 1, "question": "What is 6*7?"}).content)

### `MessagesPlaceholder`: a slot for a list of messages

Chat history, retrieved examples or an agent's scratchpad are *lists* of messages, not strings.
`MessagesPlaceholder("history")` inserts a list at that position. `optional=True` lets callers omit it.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder("history", optional=True),
    ("human", "{question}"),
])
print("required:", chat_prompt.input_variables, "| optional:", chat_prompt.optional_variables)

past = [HumanMessage("My name is Sujan."), AIMessage("Hi Sujan!")]
for m in chat_prompt.invoke({"history": past, "question": "What is my name?"}).to_messages():
    print(f"  {type(m).__name__:14s} {m.content}")
print("without history:", len(chat_prompt.invoke({"question": "hi"}).to_messages()), "messages")
print("answer:", (chat_prompt | llm).invoke({"history": past, "question": "What is my name?"}).content)

### Few-shot prompting

**Zero-shot** = instructions only. **Few-shot** = instructions plus a handful of solved examples, which is the
cheapest way to pin down format and edge cases. In a chat model the examples are best written as fake
human/assistant turns: `FewShotChatMessagePromptTemplate` renders each example through a small template.

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"review": "Arrived broken, awful packaging.", "label": "negative"},
    {"review": "It does the job, nothing special.", "label": "neutral"},
    {"review": "Absolutely love it, works great!", "label": "positive"},
    {"review": "Stopped working after two days.", "label": "negative"},
    {"review": "Fine for the price.", "label": "neutral"},
    {"review": "Best purchase this year.", "label": "positive"},
    {"review": "Customer service never replied.", "label": "negative"},
    {"review": "Setup was easy and fast.", "label": "positive"},
]
example_prompt = ChatPromptTemplate.from_messages([("human", "{review}"), ("ai", "{label}")])

def few_shot_prompt(k: int) -> ChatPromptTemplate:
    return ChatPromptTemplate.from_messages([
        ("system", "Classify the sentiment of each review as one of: positive, negative, neutral. Reply with the label only."),
        FewShotChatMessagePromptTemplate(example_prompt=example_prompt, examples=examples[:k]),
        ("human", "{review}"),
    ])

three_shot = few_shot_prompt(3)
for m in three_shot.invoke({"review": "The battery is terrible and it is slow."}).to_messages():
    print(f"  {type(m).__name__:14s} {m.content}")
label = (three_shot | llm).invoke({"review": "The battery is terrible and it is slow."}).content
print("label:", label)
assert label == "negative"

Few-shot examples are not free: they are **re-sent on every call**. The chart counts the input tokens of the same
request with 0 to 8 examples (locally, no API calls). Two to five well-chosen, diverse examples — covering the edge
cases you care about — usually beat a long list (`pr03`); beyond that you pay per call for diminishing returns, and
at some volume fine-tuning or retrieved (dynamic) examples become cheaper.

In [ ]:
ks = [0, 1, 2, 3, 4, 6, 8]
tokens = [llm.get_num_tokens_from_messages(few_shot_prompt(k).invoke({"review": "Great value."}).to_messages())
          for k in ks]
for k, t in zip(ks, tokens):
    print(f"{k} examples -> {t} input tokens")
assert tokens == sorted(tokens) and tokens[-1] > tokens[0]

plt.figure(figsize=(7, 3.2))
plt.bar([str(k) for k in ks], tokens, color="tab:purple")
plt.title("Input tokens per request vs number of few-shot examples")
plt.xlabel("examples in the prompt"); plt.ylabel("input tokens")
plt.tight_layout(); plt.show()

### Partial variables

`.partial()` fills some variables now and leaves the rest for later — handy for values that are fixed per deployment
(company name) or computed at call time (today's date, passed as a function).

In [ ]:
from datetime import date

acme_prompt = support_prompt.partial(company="Acme", language="English")
print("still needed:", acme_prompt.input_variables)

dated = ChatPromptTemplate.from_messages([("system", "Today is {today}."), ("human", "{q}")]).partial(
    today=lambda: date(2026, 9, 29).isoformat())      # a function: evaluated on every invoke
print(dated.invoke({"q": "hello"}).to_messages()[0].content)

## 7. LangChain ↔ raw OpenAI SDK

| Task | OpenAI SDK (`client = OpenAI()`) | LangChain (`llm = ChatOpenAI(...)`) |
|---|---|---|
| one call | `client.chat.completions.create(model=..., messages=[...])` | `llm.invoke([...])` |
| reply text | `resp.choices[0].message.content` | `ai.content` / `ai.text` |
| messages | dicts `{"role": "user", "content": ...}` | `HumanMessage(...)`, tuples or the same dicts |
| token usage | `resp.usage.prompt_tokens` / `completion_tokens` | `ai.usage_metadata["input_tokens"]` / `["output_tokens"]` |
| finish reason | `resp.choices[0].finish_reason` | `ai.response_metadata["finish_reason"]` |
| streaming | `create(..., stream=True)`, read `chunk.choices[0].delta.content` | `for chunk in llm.stream(...)`: `chunk.content` |
| tools | JSON schema by hand; `tool_calls[i].function.arguments` is a **JSON string** | `llm.bind_tools([python_fn])`; `ai.tool_calls[i]["args"]` is a **dict** |
| structured output | `client.chat.completions.parse(response_format=Model)` | `llm.with_structured_output(Model)` |
| many requests | your own threads / `asyncio.gather` | `llm.batch([...])`, `await llm.abatch([...])` |
| retries / fallback | `OpenAI(max_retries=2)`; fallback by hand | `ChatOpenAI(max_retries=2)`, `.with_retry()`, `.with_fallbacks([...])` |
| another provider | a different SDK and message format | another class (`ChatAnthropic`, `ChatOllama`) — same methods |

Same request both ways:

In [ ]:
messages = [{"role": "system", "content": "You are a calculator."}, {"role": "user", "content": "What is 17*23?"}]

sdk_resp = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
lc_resp = llm.invoke(messages)

print("SDK      :", sdk_resp.choices[0].message.content, "| tokens in/out:",
      sdk_resp.usage.prompt_tokens, "/", sdk_resp.usage.completion_tokens)
print("LangChain:", lc_resp.content, "| tokens in/out:",
      lc_resp.usage_metadata["input_tokens"], "/", lc_resp.usage_metadata["output_tokens"])
assert "391" in sdk_resp.choices[0].message.content and "391" in lc_resp.content

### When to use which

| Situation | Pick | Why |
|---|---|---|
| one well-understood call, or a short hand-written loop | **SDK** | nothing between you and the request; easiest to debug |
| you need provider switching, many integrations (loaders, vector stores, retrievers) | **LangChain** | the standard interfaces are the whole point |
| streaming, batching, retries, fallbacks, tracing across a multi-step pipeline | **LangChain (LCEL)** | you get them on every step for free |
| stateful agents: loops, checkpoints, human approval, resumable runs | **LangGraph** (or `create_agent`, which is built on it) | durable state is hard to hand-roll |
| a single brand-new provider feature on day one | **SDK** | frameworks lag the API by days or weeks |

The mature position: use the framework for the plumbing, keep your control flow visible, and always be able to print
the exact messages being sent (`convert_to_openai_messages`, or a tracing tool such as LangSmith).

## Try it yourself

**1. A translation template.** Build a `ChatPromptTemplate` with variables `source_lang`, `target_lang` and `text`,
partial `source_lang="English"`, and print the messages for `target_lang="French"`, `text="Good morning"`.

In [ ]:
# Solution — try it yourself first, then run this
translate = ChatPromptTemplate.from_messages([
    ("system", "Translate from {source_lang} to {target_lang}. Reply with the translation only."),
    ("human", "{text}"),
]).partial(source_lang="English")

print("still needed:", translate.input_variables)
for m in translate.invoke({"target_lang": "French", "text": "Good morning"}).to_messages():
    print(f"  {type(m).__name__:14s} {m.content}")
assert translate.input_variables == ["target_lang", "text"]

**2. Budget a chat.** Using `get_num_tokens_from_messages` and `cost_usd`, estimate the input cost of a 50-turn chat
where every user turn and every reply is the text below. How much more is it than 10 turns?

In [ ]:
# Solution — try it yourself first, then run this
def chat_input_tokens(turns: int, user: str, reply: str) -> int:
    msgs, total = [SystemMessage("You are a helpful assistant.")], 0
    for _ in range(turns):
        msgs.append(HumanMessage(user))
        total += llm.get_num_tokens_from_messages(msgs)
        msgs.append(AIMessage(reply))
    return total

t10 = chat_input_tokens(10, turn_text, reply_text)
t50 = chat_input_tokens(50, turn_text, reply_text)
for n, t in [(10, t10), (50, t50)]:
    print(f"{n} turns: {t:6d} input tokens -> ${cost_usd({'input_tokens': t, 'output_tokens': 0}):.5f}")
print(f"5x the turns costs {t50 / t10:.1f}x the input tokens")
assert t50 / t10 > 5            # super-linear: the reason to trim or summarise history

**3. Round-trip the message formats.** Convert a list of OpenAI dicts (including an assistant tool call and a tool
result) to LangChain messages with `convert_to_messages`, then back with `convert_to_openai_messages`, and check that
the roles and the tool-call id survive.

In [ ]:
# Solution — try it yourself first, then run this
from langchain_core.messages import convert_to_messages

wire = [
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "", "tool_calls": [
        {"id": "call_9", "type": "function", "function": {"name": "calculator", "arguments": '{"expression": "2+2"}'}}]},
    {"role": "tool", "tool_call_id": "call_9", "content": "4"},
]
lc_msgs = convert_to_messages(wire)
print([type(m).__name__ for m in lc_msgs], "| parsed args:", lc_msgs[1].tool_calls[0]["args"])
back = convert_to_openai_messages(lc_msgs)
print([m["role"] for m in back])
assert [m["role"] for m in back] == ["user", "assistant", "tool"]
assert back[2]["tool_call_id"] == back[1]["tool_calls"][0]["id"] == "call_9"

## Say it in an interview

- **"What does LangChain give you over the SDK?"** — A standard interface: every chat model is a Runnable with
  `invoke` / `batch` / `stream` / `ainvoke`, typed messages, prompt templates, tool binding from Python functions,
  structured output, retries and fallbacks, and hundreds of integrations. The cost is an abstraction between you and
  the request, so I always keep a way to print the exact messages sent.
- **"LangChain vs LangGraph vs LlamaIndex?"** — LangChain is components + integrations; LangGraph is the stateful
  runtime (checkpoints, interrupts, loops) that LangChain 1.x's `create_agent` is built on; LlamaIndex goes deepest on
  ingestion and retrieval.
- **Roles:** system = standing rules, user = request, assistant = model output incl. tool calls, tool = your result
  for one tool-call id. Missing a tool result is a 400, not a warning.
- **Templates:** every prompt goes through a template so it can be versioned, tested and diffed; `MessagesPlaceholder`
  for lists (history, examples); `.partial()` for per-deployment values; double the braces for literal JSON.
- **Few-shot:** 2–5 diverse examples that cover the edge cases; they are paid for on every call, so measure the tokens.
- **Cost:** read `usage_metadata`; cost = input × input price + output × output price (cached input cheaper). A chat
  resends its history, so total tokens grow ~quadratically with turns — trim, summarise or retrieve history.
- **Streaming:** improves time to first token, not total latency. Check `finish_reason == "length"` for truncation.
- **Traps:** 0.x imports (`langchain.chains`, `langchain.memory`, `AgentExecutor`) fail in 1.x — they live in
  `langchain-classic`; `temperature` is rejected by reasoning models; the model is stateless.

## Next

- [02 · LCEL, Runnables and output parsers](02_lcel_runnables_and_output_parsers.ipynb) — the `|` operator, parallel
  steps, routing, retries, batching.
- [03 · Structured output and tools](03_structured_output_and_tools.ipynb)
- The same ideas without a framework: [OpenAI API basics](../08_llm_apps/01_openai_api_basics.ipynb),
  [prompt engineering](../08_llm_apps/02_prompt_engineering.ipynb),
  [conversation memory and context windows](../08_llm_apps/06_conversation_memory_and_context_windows.ipynb)
- Theory: [LangChain course](../../langchain/index.html) (chapters 1–3) · [interview bank](../../ai_interview_prep/index.html)